# 개별종목 조합G — RandomForest

`기본모델/02.RandomForest.ipynb`과 같은 `models.random_forest.build_random_forest_baseline`을 가져오고
조합G 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.random_forest import build_random_forest_baseline  # noqa: E402

MODEL_NAME = 'RandomForest'
MODEL_BUILDER = build_random_forest_baseline


In [2]:
# 2. 조합G의 피처 값만 지정합니다.
import json

COMBINATION = 'G'
FEATURE_COLUMNS = (
    'dist_high_60',
    'sma_gap_20_60',
    'relative_ret_5_market',
    'rsi_14',
    'hv_20',
    'turnover_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합G 피처: ('dist_high_60', 'sma_gap_20_60', 'relative_ret_5_market', 'rsi_14', 'hv_20', 'turnover_20')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4031,0.5012,-0.0981,0.3472,0.3476,0.0258,0.3617,0.2387,0.3141
1,2,NaN,980,20150123,20150421,0.3688,0.3978,-0.0291,0.3408,0.3507,0.0315,0.3543,0.2765,0.3239
2,3,balanced,1210,20151228,20160328,0.3481,0.3762,-0.0281,0.3444,0.3441,0.0197,0.3488,0.3024,0.3303
3,4,balanced,1439,20161202,20170228,0.3975,0.4617,-0.0643,0.3644,0.3643,0.0531,0.3837,0.2538,0.3261
4,5,balanced,1669,20171113,20180207,0.3548,0.3901,-0.0353,0.3477,0.3475,0.0228,0.3507,0.3326,0.3448
5,6,balanced,1899,20181024,20190118,0.3945,0.3725,0.0220,0.3918,0.3917,0.0904,0.4030,0.3605,0.3816
6,7,balanced,2129,20190930,20191224,0.4092,0.4781,-0.0690,0.3649,0.3700,0.0646,0.3801,0.3353,0.3673
7,8,balanced,2359,20200902,20201130,0.3449,0.3476,-0.0027,0.3437,0.3502,0.0267,0.3566,0.4233,0.3671
8,9,balanced,2589,20210806,20211105,0.3413,0.3914,-0.0501,0.3324,0.3354,0.0037,0.3554,0.3281,0.3338
9,10,balanced,2818,20220714,20221012,0.3547,0.3454,0.0093,0.3530,0.3552,0.0333,0.3586,0.3226,0.3428


,OOS 폴드 평균
accuracy,0.3728
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0241
macro_f1,0.3567
balanced_accuracy,0.3590
mcc,0.0417
pr_auc_macro_ovr,0.3677
down_recall,0.3320
core_harmonic_mean,0.3502


재실행 명령: python scripts/run_stock_model_experiment.py
